# Neandertales Caffe — Class 3
## From Transactional Data to Analytical Data

Last class we created new columns — bag_size, month, quarter, price_tier.

Today:
1. **Recap** — write from memory
2. **Dimension tables** — products and customers
3. **Joins** — connect tables using IDs
4. **Analytical layer** — agg(), pivot_table()
5. **Deep analysis** — questions impossible without enriched data

> *"Transactional data records what happened. Analytical data is shaped so you can ask why."*

# 0. Setup
Run everything here before starting — this recreates the columns from last class.

In [ ]:
# ── Google Colab ───────────────────────────────────────────
# from google.colab import files
# files.upload()

# ── VS Code ─────────────────────────────────────────────────
# Make sure neandertales_orders.csv is in the same folder

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv("neandertales_orders.csv")

# Revenue column
orders["revenue"] = orders["price"] * orders["quantity"]

# Bag size
def extract_bag_size(product):
    if "500g" in product:
        return "500g"
    elif "1000g" in product:
        return "1000g"
    else:
        return "No size"

orders["bag_size"] = orders["product"].apply(extract_bag_size)

# Date columns
orders["date"]       = pd.to_datetime(orders["date"])
min_date             = orders["date"].min()
max_date             = orders["date"].max()

calendar             = pd.DataFrame({"date": pd.date_range(start=min_date, end=max_date, freq="D")})
calendar["year"]     = calendar["date"].dt.to_period("Y")
calendar["quarter"]  = calendar["date"].dt.to_period("Q")
calendar["year_month"]= calendar["date"].dt.to_period("M")
calendar["month_name"]= calendar["date"].dt.month_name()
calendar["day_name"] = calendar["date"].dt.day_name()
calendar["is_weekend"]= calendar["date"].dt.day_of_week >= 5

orders = orders.merge(calendar, on="date", how="left")

# Price tier
def classify_price(price):
    if price < 15:
        return "Low"
    elif price <= 35:
        return "Mid"
    else:
        return "High"

orders["price_tier"]     = orders["price"].apply(classify_price)
orders["is_large_order"] = orders["quantity"] >= 3

print("Dataset ready!")
print("Shape:", orders.shape)
print("Columns:", orders.columns.tolist())

# 1. Recap — Write From Memory
Minimal hints. No examples shown.

## 1.1 — Total revenue and average revenue per order

In [ ]:
# Hint: df["col"].sum() and df["col"].mean()
total_revenue = orders["_______"]._____()
avg_revenue   = orders["_______"]._____()

print("Total revenue:", round(total_revenue, 2), "euros")
print("Average revenue per row:", round(avg_revenue, 2), "euros")

## 1.2 — Revenue per category, sorted highest first

In [ ]:
# Hint: df.groupby("col")["col"].sum().sort_values(ascending=False)
rev_by_cat = orders.groupby("________")["_______"]._____().round(2).sort_values(ascending=False)

print(rev_by_cat)

## 1.3 — How many 500g vs 1000g orders?

In [ ]:
# Hint: df["col"].value_counts()
size_counts = orders["________"].___________()

print(size_counts)

## 1.4 — Revenue per quarter, bar chart

In [ ]:
# Step 1 — calculate
# Hint: df.groupby("col")["col"].sum()
rev_per_quarter = orders.groupby("_______")["_______"]._____().round(2)

# Step 2 — convert index for plotting
rev_per_quarter.index = rev_per_quarter.index.astype(str)

# Step 3 — plot
rev_per_quarter.plot(kind="bar", color="steelblue")

plt.title("___")
plt.xlabel("Quarter")
plt.ylabel("___")
plt.xticks(rotation=___)

plt.tight_layout()
plt.show()

# 3. Dimension Tables — Building a Proper Data Model
Right now our dataset is a **fact table** — every row is one transaction.

In real databases you split data into:
- **Fact table** — the transactions (what we have now)
- **Dimension tables** — reference data about products and customers

> *"A fact table records events. A dimension table describes the participants."*

## 3.1 — Product Dimension Table

In [ ]:
# Product dimension table — one row per product
# We join on product_id — NOT the product name
# Joining on IDs is safer — a typo in a name would break the join silently

products_dim = pd.DataFrame({
    "product_id": [
        "P001","P002","P003","P004","P005","P006","P007","P008",
        "P009","P010","P011","P012","P013","P014","P015"
    ],
    "product": [
        "Espresso Blend 500g",          "Espresso Blend 1000g",
        "Colombia Single Origin 500g",  "Colombia Single Origin 1000g",
        "Ethiopia Natural 500g",        "Ethiopia Natural 1000g",
        "Dark Roast Blend 500g",        "Dark Roast Blend 1000g",
        "Classic Mug 300ml",            "Travel Mug 400ml",
        "Filters V60 x100",             "Cleaning Tablets x10",
        "French Press 600ml",           "Moka Pot",
        "Coffee Grinder Manual"
    ],
    "category": [
        "Coffee","Coffee","Coffee","Coffee","Coffee","Coffee","Coffee","Coffee",
        "Accessories","Accessories","Accessories","Accessories",
        "Equipment","Equipment","Equipment"
    ],
    "bag_size": [
        "500g","1000g","500g","1000g","500g","1000g","500g","1000g",
        "No size","No size","No size","No size","No size","No size","No size"
    ],
    "roast": [
        "Medium","Medium","Light","Light",
        "Natural","Natural","Dark","Dark",
        None,None,None,None,None,None,None
    ],
    "origin": [
        "Blend","Blend","Colombia","Colombia",
        "Ethiopia","Ethiopia","Blend","Blend",
        None,None,None,None,None,None,None
    ],
    "base_price": [
        12.90,22.90,14.90,27.90,15.90,29.90,11.90,21.90,
        9.90,24.90,7.90,6.90,34.90,29.90,44.90
    ]
})

print("Product dimension table:")
print(products_dim.to_string(index=False))

### Your turn — how many products have a known roast?

In [ ]:
# Your turn — how many products have a known roast?
# Hint: df["col"].notna().sum()
known_roast = products_dim["_____"]._______().sum()

print("Products with a known roast:", known_roast)
print()

# Your turn — how many unique product_ids?
# Hint: df["col"].nunique()
unique_ids = products_dim["__________"].________()
print("Unique product IDs:", unique_ids)

## 3.2 — Customer Dimension Table

In [ ]:
# Customer dimension table — one row per customer
# Same customer_id values as the fact table — this is what makes the join work

customers_dim = pd.DataFrame({
    "customer_id": [
        "C001","C002","C003","C004","C005","C006","C007","C008","C009","C010",
        "C011","C012","C013","C014","C015","C016","C017","C018","C019","C020",
        "C021","C022","C023","C024","C025","C026","C027","C028","C029","C030"
    ],
    "customer": [
        "Alice","Bob","Carol","David","Eva","Frank","Grace","Henry","Iris","James",
        "Kate","Liam","Mia","Noah","Olivia","Paul","Quinn","Rosa","Sam","Tina",
        "Uma","Victor","Wendy","Xander","Yara","Zoe","Marco","Nina","Oscar","Paula"
    ],
    "city": [
        "Berlin","Berlin","Munich","Berlin","Cologne","Berlin","Hamburg","Munich",
        "Cologne","Berlin","Munich","Munich","Berlin","Berlin","Munich","Berlin",
        "Hamburg","Berlin","Munich","Munich","Cologne","Berlin","Berlin","Frankfurt",
        "Berlin","Cologne","Hamburg","Berlin","Frankfurt","Frankfurt"
    ],
    "email": [
        "alice@gmail.com","bob@outlook.com","carol@web.de","david@gmx.de",
        "eva@yahoo.com","frank@icloud.com","grace@gmail.com","henry@web.de",
        "iris@gmx.de","james@outlook.com","kate@gmail.com","liam@outlook.com",
        "mia@web.de","noah@gmx.de","olivia@yahoo.com","paul@icloud.com",
        "quinn@gmail.com","rosa@web.de","sam@gmx.de","tina@outlook.com",
        "uma@gmail.com","victor@outlook.com","wendy@web.de","xander@gmx.de",
        "yara@yahoo.com","zoe@icloud.com","marco@gmail.com","nina@web.de",
        "oscar@gmx.de","paula@outlook.com"
    ],
    "join_date": [
        "2023-02-14","2023-03-01","2023-01-20","2022-11-05","2023-05-18",
        "2022-08-30","2023-06-12","2022-12-01","2023-04-22","2022-09-15",
        "2023-07-08","2023-02-28","2022-10-17","2023-08-05","2022-07-19",
        "2023-09-11","2023-01-30","2022-06-25","2023-10-03","2022-05-14",
        "2023-11-20","2022-04-07","2023-12-01","2022-03-22","2024-01-09",
        "2022-02-18","2024-02-14","2022-01-05","2024-03-20","2023-03-15"
    ],
    "segment": [
        "Regular","Occasional","Regular","Regular","Occasional",
        "VIP","Regular","Occasional","Regular","VIP",
        "Regular","Regular","VIP","Occasional","Regular",
        "Occasional","Regular","VIP","Regular","Occasional",
        "Regular","Occasional","Regular","Regular","VIP",
        "Regular","Occasional","Regular","VIP","Regular"
    ],
    "preferred_coffee": [
        "Espresso Blend","Dark Roast Blend","Colombia Single Origin","Espresso Blend",
        "Ethiopia Natural","Colombia Single Origin","Dark Roast Blend","Espresso Blend",
        "Ethiopia Natural","Espresso Blend","Colombia Single Origin","Dark Roast Blend",
        "Espresso Blend","Ethiopia Natural","Colombia Single Origin","Espresso Blend",
        "Dark Roast Blend","Colombia Single Origin","Ethiopia Natural","Espresso Blend",
        "Colombia Single Origin","Dark Roast Blend","Espresso Blend","Ethiopia Natural",
        "Colombia Single Origin","Espresso Blend","Dark Roast Blend","Ethiopia Natural",
        "Espresso Blend","Colombia Single Origin"
    ]
})

# Convert join_date to datetime
customers_dim["join_date"] = pd.to_datetime(customers_dim["join_date"])

print("Customer dimension table:")
print(customers_dim.head(10).to_string(index=False))
print()
print(f"Total customers: {len(customers_dim)}")
print()
print("Segments:")
print(customers_dim["segment"].value_counts())
print()
print("Preferred coffee:")
print(customers_dim["preferred_coffee"].value_counts())

### Your turn — how many VIP customers do we have?

In [ ]:
# Hint: df[df["col"] == "value"]
vip_customers = _______[___________["_______"] == "______"]

vip_count = len(vip_customers)
print("VIP customers:", vip_count)

## 3.3 — Joining the Dimension Tables

In [ ]:
# Example — join orders with product dimension using product_id
# Why product_id and NOT product name?
# Because IDs are guaranteed unique
# A typo in a product name would break the join silently

# Check that product_id exists in both tables before joining
print("Fact table — product_id sample:")
print(orders["product_id"].head(5).tolist())
print()
print("Dimension — product_id sample:")
print(products_dim["product_id"].head(5).tolist())
print()

# Now join on product_id
orders_enriched = orders.merge(
    products_dim[["product_id", "roast", "origin", "bag_size", "base_price"]],
    on="product_id",
    how="left"
)

print("After joining product dimension:")
print(orders_enriched[["product_id", "product", "roast", "origin", "bag_size"]].head(8))

### Your turn — join orders_enriched with customer dimension

In [ ]:
# Your turn — join orders_enriched with customer dimension
# Hint: df.merge(other_df, on="shared_col", how="left")
# Use customer_id — always join on IDs, not names

orders_enriched = orders_enriched.merge(
    customers_dim[["customer_id", "segment", "preferred_coffee"]],
    on="___________",
    how="____"
)

print("Columns now available:")
print(orders_enriched.columns.tolist())
print()
print("Missing segment values:", orders_enriched["segment"].isnull().sum())

### Your turn — revenue per customer segment

In [ ]:
# Hint: df.groupby("col")["col"].sum().sort_values(ascending=False)
rev_by_segment = orders_enriched.groupby("_______")["_______"]._____().round(2)

rev_by_segment_sorted = rev_by_segment.sort_values(ascending=False)
print(rev_by_segment_sorted)

### Your turn — which coffee roast generates the most revenue?

In [ ]:
# Step 1 — filter Coffee only
# Hint: df[df["col"] == "value"]
coffee_e = orders_enriched[orders_enriched["________"] == "Coffee"]

# Step 2 — groupby roast, sum revenue
# Hint: df.groupby("col")["col"].sum().sort_values(ascending=False)
roast_rev = coffee_e.groupby("_____")["_______"]._____().round(2)

roast_rev_sorted = roast_rev.sort_values(ascending=False)
print(roast_rev_sorted)

# 4. The Analytical Layer
Now that the data is enriched, we build summary tables
that could power a real dashboard.

> *"Transactional: one row per event.
> Analytical: one row per dimension you care about."*

## 4.1 — Monthly Summary Table

In [ ]:
# Example — build a monthly summary with multiple metrics at once
# agg() lets you calculate several things in one groupby
monthly_summary = orders_enriched.groupby("year_month").agg(
    total_revenue = ("revenue",  "sum"),
    total_orders  = ("order_id", "nunique"),
    total_units   = ("quantity", "sum"),
    avg_revenue   = ("revenue",  "mean")
).round(2)

# Convert index for display
monthly_summary.index = monthly_summary.index.astype(str)

print(monthly_summary)

### Your turn — build a city summary table
For each city show: total_revenue, total_orders, avg_revenue

In [ ]:
# Hint: same structure as above but groupby "city"
# df.groupby("col").agg(new_col=("col","function"), ...)
city_summary = orders_enriched.groupby("____").agg(
    total_revenue = ("_______", "___"),
    total_orders  = ("________", "______"),
    avg_revenue   = ("_______", "____")
).round(2)

city_summary_sorted = city_summary.sort_values("total_revenue", ascending=False)
print(city_summary_sorted)

## 4.2 — Revenue Matrix (pivot table)

In [ ]:
# Example — city vs category revenue matrix
# pivot_table: rows=index, columns=columns, values=values
city_cat_matrix = orders_enriched.pivot_table(
    values  = "revenue",
    index   = "city",
    columns = "category",
    aggfunc = "sum"
).round(2)

print(city_cat_matrix)

### Your turn — segment vs category revenue matrix

In [ ]:
# Hint: same as above but use "segment" as index
seg_cat_matrix = orders_enriched.pivot_table(
    values  = "revenue",
    index   = "________",
    columns = "________",
    aggfunc = "___"
).round(2)

print(seg_cat_matrix)

### Your turn — month vs bag_size revenue matrix (Coffee only)

In [ ]:
# Step 1 — filter Coffee only
coffee_e = orders_enriched[orders_enriched["________"] == "Coffee"]

# Step 2 — pivot table: year_month vs bag_size
# Hint: pivot_table with index="year_month", columns="bag_size"
size_month_matrix = coffee_e.pivot_table(
    values  = "revenue",
    index   = "________",
    columns = "________",
    aggfunc = "___"
).round(2)

# Convert index for display
size_month_matrix.index = size_month_matrix.index.astype(str)

print(size_month_matrix)

## 4.3 — Order-level Aggregation

In [ ]:
# Order-level aggregation
# The fact table has one row per product per order
# This gives us one row per order — the full transaction

order_summary = orders.groupby(["order_id", "customer", "city"]).agg(
    total_items   = ("quantity", "sum"),
    total_revenue = ("revenue",  "sum"),
    num_products  = ("product",  "nunique"),
).round(2).reset_index()

print("Order summary — first 10 rows:")
print(order_summary.head(10))

In [ ]:
# How many rows does the fact table have?
fact_rows = len(orders)
print("Fact table rows:", fact_rows)

# How many unique orders does the summary have?
order_rows = len(order_summary)
print("Order summary rows:", order_rows)

print()
print("The difference: fact table has one row per product")
print("Order summary has one row per order")

In [ ]:
# Average order value
# Hint: df["col"].mean()
avg_order_value = order_summary["total_revenue"].mean()

print("Average order value:", round(avg_order_value, 2), "euros")

In [ ]:
# Most valuable order
# Hint: df["col"].max()
max_order_value = order_summary["total_revenue"].max()

print("Most valuable order:", max_order_value, "euros")

In [ ]:
# Orders with more than one product
# Hint: df[df["col"] > value]
multi_product = order_summary[order_summary["num_products"] > 1]

multi_product_count = len(multi_product)
print("Orders with more than one product:", multi_product_count)

Your turn — what percentage of orders had more than one product?

In [ ]:
# Step 1 — total number of orders
total_orders = len(order_summary)

# Step 2 — calculate percentage
# Hint: count / total * 100
multi_pct = multi_product_count / ____________ * 100

# Step 3 — print
print("Percentage of multi-product orders:", round(multi_pct, 1), "%")

Your turn — bar chart of how many products per order

In [ ]:
# Step 1 — count orders by num_products
# Hint: df["col"].value_counts().sort_index()
products_per_order = order_summary["___________"].value_counts().sort_index()

# Step 2 — plot
products_per_order.plot(kind="bar", color="steelblue")

plt.title("How Many Products per Order?")
plt.xlabel("Number of different products")
plt.ylabel("Number of orders")
plt.xticks(rotation=___)

plt.tight_layout()
plt.show()

# 5. Deep Analysis
Now that the data is properly shaped we can answer questions
that were impossible with the raw data.

## 5.1 — Do VIP customers buy differently?

In [ ]:
# Example — compare VIP vs Regular by category
# Hint: df[df["col"] == "value"]
vip     = orders_enriched[orders_enriched["segment"] == "VIP"]
regular = orders_enriched[orders_enriched["segment"] == "Regular"]

vip_by_cat     = vip.groupby("category")["revenue"].sum().sort_values(ascending=False).round(2)
regular_by_cat = regular.groupby("category")["revenue"].sum().sort_values(ascending=False).round(2)

print("VIP — revenue by category:")
print(vip_by_cat)
print()
print("Regular — revenue by category:")
print(regular_by_cat)

### Your turn — compare VIP vs Regular by average order value

In [ ]:
# Step 1 — average revenue per row for VIP
# Hint: df["col"].mean()
vip_avg = vip["_______"]._____()

# Step 2 — average revenue per row for Regular
regular_avg = regular["_______"]._____()

# Step 3 — print both
print("VIP average revenue per row:    ", round(vip_avg, 2), "euros")
print("Regular average revenue per row:", round(regular_avg, 2), "euros")

## 5.2 — Customer loyalty — who ordered in multiple quarters?

In [ ]:
# Count how many different quarters each customer ordered in
# Hint: df.groupby("col")["col"].nunique()
quarters_per_customer = orders_enriched.groupby("customer_id")["_______"].________()

quarter_dist = quarters_per_customer.value_counts().sort_index()
print("Quarters active per customer:")
print(quarter_dist)

### Your turn — find customers who ordered in all 4 quarters

In [ ]:
# Step 1 — filter customers who ordered in all 4 quarters
# Hint: series[series == value]
loyal = quarters_per_customer[quarters_per_customer == ___]

loyal_count = len(loyal)
print("Loyal customers (all 4 quarters):", loyal_count)

# Step 2 — find their names
# Hint: df[df["col"].isin(list)]
loyal_ids   = loyal.index.tolist()
loyal_names = orders_enriched[orders_enriched["customer_id"].isin(loyal_ids)]["customer"].unique()

print("Names:", loyal_names)

## 5.3 — 500g vs 1000g by month — chart

In [ ]:
# Filter coffee only
coffee_e = orders_enriched[orders_enriched["category"] == "Coffee"]

# Group by year_month and bag_size, sum revenue, unstack
size_by_month = coffee_e.groupby(["year_month","bag_size"])["revenue"].sum().unstack(fill_value=0)
size_by_month.index = size_by_month.index.astype(str)

# Plot grouped bar chart
size_by_month.plot(kind="bar", figsize=(12, 4), edgecolor="white")

plt.title("Coffee Revenue: 500g vs 1000g by Month")
plt.xlabel("Month")
plt.ylabel("Revenue (euros)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Bag Size")

plt.tight_layout()
plt.show()

### Your turn — same chart but for segment instead of bag_size

In [ ]:
# Step 1 — groupby year_month and segment, sum revenue, unstack
# Hint: df.groupby(["col1","col2"])["col"].sum().unstack(fill_value=0)
seg_by_month = orders_enriched.groupby(["year_month","________"])["revenue"].sum()
seg_by_month.index = seg_by_month.index.astype(str)

# Step 2 — plot
seg_by_month.plot(kind="bar", figsize=(12, 4), edgecolor="white")

plt.title("___")
plt.xlabel("Month")
plt.ylabel("Revenue (euros)")
plt.xticks(rotation=___, ha="right")
plt.legend(title="___")

plt.tight_layout()
plt.show()

# 7. Challenges
Use `orders_enriched` which has all the new columns.
Small hint for each challenge.

## Challenge 1 — Best month for each category
Which month had the highest revenue for Coffee? For Equipment? For Accessories?

**Hint:** groupby `month` and `category`, sum `revenue`, then filter by category

In [ ]:
# Your code here


## Challenge 2 — 500g vs 1000g revenue share
Of all coffee revenue, what percentage comes from 500g and what from 1000g?

**Hint:** filter Coffee, groupby `bag_size`, sum revenue, divide by total coffee revenue × 100

In [ ]:
# Your code here


## Challenge 3 — Top city per category
For each category, which city generates the most revenue?

**Hint:** groupby `city` and `category`, sum revenue, then find the max per category

In [ ]:
# Your code here


## Challenge 4 — Customer lifetime value table
Create a summary with: customer name, total revenue, number of orders, average order value.

**Hint:** groupby `customer`, agg revenue (sum) and order_id (nunique), then divide for average

In [ ]:
# Your code here


## Challenge 5 — Colored horizontal bar chart
Total revenue per product as a horizontal bar chart.
Color Coffee bars blue, Accessories orange, Equipment green.

**Hint:** build a color list by mapping each product's category to a color, then pass to `.plot(color=...)`

In [ ]:
# Your code here


## Challenge 6 — Your own question
Pick one question we have not answered yet and answer it with code and a chart.

**Ideas:**
- Which month had the highest Equipment revenue?
- Do large orders (is_large_order = True) come from specific cities?
- Which origin coffee (Colombia, Ethiopia, Blend) is most popular by units sold?

In [ ]:
# My question:
# ___________________________________________

# My code:


# My conclusion:
# ___________________________________________

# Answers — only open after you have tried!

In [ ]:
# Challenge 1 — best month per category
monthly_cat = orders_enriched.groupby(["month","category"])["revenue"].sum().round(2)

for cat in ["Coffee","Accessories","Equipment"]:
    cat_data   = monthly_cat.xs(cat, level="category")
    best_month = cat_data.idxmax()
    best_rev   = cat_data.max()
    print(f"{cat}: best month = {best_month}  (€{best_rev})")

In [ ]:
# Challenge 2 — 500g vs 1000g revenue share
coffee_e     = orders_enriched[orders_enriched["category"] == "Coffee"]
size_revenue = coffee_e.groupby("bag_size")["revenue"].sum()
total_coffee = size_revenue.sum()
size_share   = (size_revenue / total_coffee * 100).round(1)

print("Coffee revenue share by bag size:")
print(size_share)

In [ ]:
# Challenge 3 — top city per category
city_cat = orders_enriched.groupby(["category","city"])["revenue"].sum().round(2)

for cat in ["Coffee","Accessories","Equipment"]:
    cat_data = city_cat.xs(cat, level="category")
    top_city = cat_data.idxmax()
    top_rev  = cat_data.max()
    print(f"{cat}: {top_city}  (€{top_rev})")

In [ ]:
# Challenge 4 — customer lifetime value
clv = orders_enriched.groupby("customer").agg(
    total_revenue = ("revenue",  "sum"),
    num_orders    = ("order_id", "nunique"),
).round(2)
clv["avg_order_value"] = (clv["total_revenue"] / clv["num_orders"]).round(2)
print(clv.sort_values("total_revenue", ascending=False).head(10))

In [ ]:
# Challenge 5 — colored horizontal bar chart
prod_rev  = orders_enriched.groupby(["product","category"])["revenue"].sum().sort_values()
color_map = {"Coffee":"steelblue","Accessories":"coral","Equipment":"lightgreen"}
colors    = [color_map[cat] for cat in prod_rev.index.get_level_values("category")]

prod_rev.plot(kind="barh", color=colors)

from matplotlib.patches import Patch
legend = [Patch(color=v, label=k) for k, v in color_map.items()]
plt.legend(handles=legend)

plt.title("Neandertales Caffe — Revenue by Product", fontweight="bold")
plt.xlabel("Total Revenue (euros)")
plt.tight_layout()
plt.show()